# 029 — HiPace80Bus Function Test

First real-hardware verification of the `HiPace80Bus` class (loadlock turbo, TC80 + OmniControl + gauge).

**Usage:** set the COM port and addresses in the config cell, then run cells one by one and check each response.

**Note (KB, memory/devices/pfeiffer.md):** HiPace80 RS-485 addresses are UNVERIFIED — class defaults are OmniControl = 1, TC80 = 2, gauge = 122 (fleet-wide guess). If queries time out, the address is likely wrong (compare the HiPace300 lesson: OmniControl was at 101, not 1).

## 1. Imports

In [1]:
import sys
import os
import time

# Add path to src modules
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

#from devices.pfeiffer.hipacebus import HiPace80Bus
from devices.pfeiffer.hipacebus import HiPace300Bus

from devices.pfeiffer import pfeifferVacuumProtocol as pvp
pvp.enable_valid_char_filter()

print("HiPace80Bus imported")

HiPace80Bus imported


In [2]:
from devices.pfeiffer.hipacebus import HiPace300Bus

In [3]:
# HiPace450 main turbo pump (placeholder - needs device class implementation)
HiPace450_dict = {
    'com_port': 'COM27',
    'device_address': 101,
    'omnicontrol_address': 101,
    'tc400_address': 1,
    'gauge1_address': 122,
    'device_type': 'HiPace450',
    'description': 'Main high vacuum pump (initialized as HiPace300Bus)'
}

In [4]:
# Initialize and connect HiPace450 (using HiPace300Bus class)
hipace450 = HiPace300Bus(
    device_id="hipace450",
    port=HiPace450_dict['com_port'],
    device_address=HiPace450_dict['device_address'],
    omnicontrol_address=HiPace450_dict['omnicontrol_address'],
    tc400_address=HiPace450_dict['tc400_address'],
    gauge1_address=HiPace450_dict['gauge1_address']
)
hipace450.connect()
print(f"✅ HiPace450 connected on {HiPace450_dict['com_port']} (initialized as HiPace300Bus)")

✅ HiPace450 connected on COM27 (initialized as HiPace300Bus)


In [2]:
from devices.pfeiffer.hipacebus import HiPace80DCU

pump = HiPace80DCU("HiPace80_MPI", port="COM28", tc80_address=1, timeout=0.3)
pump.connect()

True

In [3]:
pump.get_cfg_acc_a1()

ValueError: gauge response too short to be valid len(r) = 0

In [9]:
pump.get_cfg_acc_b1()

2

In [12]:
pump._get_accessory_config_tc80(24)

0

In [11]:
pump.acknowledge_error()

In [5]:
hipace450.get_pump_status()

{'actual_speed_hz': 2,
 'actual_speed_rpm': 119,
 'set_speed_hz': 1100,
 'drive_current': 0.0,
 'drive_voltage': 23.22,
 'drive_power': 0,
 'electronics_temp': 39,
 'pump_bottom_temp': 28,
 'bearing_temp': 28,
 'target_speed_reached': False,
 'pump_accelerating': False,
 'operating_hours': 234}

In [13]:
print("Cfg accessory A1        :", hipace450.get_cfg_acc_a1())
print("Cfg accessory B1        :", hipace450.get_cfg_acc_b1())
print("Cfg accessory A2        :", hipace450.get_cfg_acc_a2())
print("Cfg accessory B2        :", hipace450.get_cfg_acc_b2())

Cfg accessory A1        : 0
Cfg accessory B1        : 1
Cfg accessory A2        : 0
Cfg accessory B2        : 0


In [12]:
hipace450.set_cfg_acc_a1(0)
hipace450.set_cfg_acc_b1(1)
hipace450.set_cfg_acc_a2(0)
hipace450.set_cfg_acc_b2(0)

In [22]:
print("Vent enabled            :", hipace450.get_vent_enabled())
print("Vent mode               :", hipace450.get_vent_mode(), "(0=delayed, 1=none, 2=direct; TC80 default 2)")

Vent enabled            : False
Vent mode               : 2 (0=delayed, 1=none, 2=direct; TC80 default 2)


In [42]:
hipace450.get_pump_status()

{'actual_speed_hz': 92,
 'actual_speed_rpm': 5558,
 'set_speed_hz': 1100,
 'drive_current': 0.0,
 'drive_voltage': 23.22,
 'drive_power': 0,
 'electronics_temp': 42,
 'pump_bottom_temp': 29,
 'bearing_temp': 30,
 'target_speed_reached': False,
 'pump_accelerating': False,
 'operating_hours': 234}

In [32]:
print("Drive current           :", hipace450.get_drive_current(), "A")
print("Drive voltage           :", hipace450.get_drive_voltage(), "V")
print("Drive power             :", hipace450.get_drive_power(), "W")
print("Temp electronics        :", hipace450.get_electronics_temperature(), "degC")
print("Temp pump bottom        :", hipace450.get_pump_bottom_temperature(), "degC")
print("Overtemp electronics    :", hipace450.is_overtemperature_electronics())
print("Overtemp pump           :", hipace450.is_overtemperature_pump())
print("Operating hours pump    :", hipace450.get_operating_hours_pump(), "h")
print("Operating hours elec    :", hipace450.get_operating_hours_electronics(), "h")
print("Pump cycles             :", hipace450.get_pump_cycles())
print("Accel/decel             :", hipace450.get_acceleration_deceleration(), "rpm/s")

Drive current           : 6.96 A
Drive voltage           : 22.64 V
Drive power             : 144 W
Temp electronics        : 41 degC
Temp pump bottom        : 28 degC
Overtemp electronics    : False
Overtemp pump           : False
Operating hours pump    : 234 h
Operating hours elec    : 1025 h
Pump cycles             : 138
Accel/decel             : 300 rpm/s


In [27]:
# START pump station
hipace450.enable_motor_pump()
hipace450.enable_pumpStatn()
# print("pump station enabled:", pump.get_pumpStatn_enabled())

In [35]:
# STOP pump station (close the LL HiPace80 valve first!)
hipace450.disable_pumpStatn()
# print("pump station enabled:", pump.get_pumpStatn_enabled())

In [39]:
# Other state changes (uncomment as needed):
hipace450.enable_vent()              # / pump.disable_vent()

In [43]:
hipace450.disable_vent()

## 2. Configuration — EDIT HERE

In [2]:
COM_PORT = 'COM35'          # <-- set the COM port here (check Device Manager)

OMNICONTROL_ADDRESS = 100    # class default 1 — UNVERIFIED on HiPace80 hardware
TC80_ADDRESS = 1           # class default 2 — UNVERIFIED on HiPace80 hardware
GAUGE1_ADDRESS = 122       # fleet-wide guess from lab_config — VERIFY (sensor is connected)

DEVICE_ID = "HiPace80_test"

## 3. Create device and connect

In [3]:
pump = HiPace80Bus(
    device_id='HiPace80_test',
    port='COM35',
    device_address=101,
    tc80_address=1,
    gauge1_address=122
)

connected = pump.connect()
print(f"connect() -> {connected}")

connect() -> True


In [4]:
pump.get_pump_status()


{'actual_speed_hz': 0,
 'actual_speed_rpm': 0,
 'set_speed_hz': 1500,
 'drive_current': 0.0,
 'drive_voltage': 24.08,
 'drive_power': 0,
 'electronics_temp': 31,
 'pump_bottom_temp': 26,
 'power_stage_temp': 30,
 'rotor_temp': 25,
 'target_speed_reached': False,
 'pump_accelerating': False,
 'operating_hours': 5}

In [8]:
pump.disconnect()

True

In [5]:
pump.get_pump_firmware_version()

'010500'

In [6]:
pump.get_pump_firmware_version()

'010500'

In [7]:
pump.get_pump_firmware_version()

'010500'

In [8]:
pump.get_pump_firmware_version()

'010500'

In [9]:
pump.get_pump_firmware_version()

'010500'

In [15]:
pump.disconnect()

True

## 4. Identify devices on the bus

First real test: if these answer, addresses and wiring are right.

In [10]:
# OmniControl (address = OMNICONTROL_ADDRESS)
print("OmniControl device name :", pump.get_omni_device_name())
print("OmniControl firmware    :", pump.get_omni_firmware_version())
print("OmniControl hardware    :", pump.get_omni_hardware_version())
print("OmniControl serial no.  :", pump.get_omni_serial_number())
print("OmniControl RS485 addr  :", pump.get_omni_rs485_address())
print("OmniControl error code  :", pump.get_omni_error_code())

OmniControl device name : OmC200
OmniControl firmware    : 010300
OmniControl hardware    : 010100
OmniControl serial no.  : 80116972
OmniControl RS485 addr  : 100
OmniControl error code  : 000000


In [11]:
# TC80 drive electronics (address = TC80_ADDRESS)
print("Pump device name        :", pump.get_pump_device_name())
print("Pump firmware           :", pump.get_pump_firmware_version())
print("Pump hardware version   :", pump.get_pump_hardware_version())
print("Pump RS485 addr         :", pump.get_rs485_address())
print("Pump error code         :", pump.get_pump_error_code())
print("Pump identification     :", pump.get_pump_identification())

Pump device name        : TC 80
Pump firmware           : 010500
Pump hardware version   : 010200
Pump RS485 addr         : 1
Pump error code         : 000000
Pump identification     : 30821


## 5. TC80 status queries (read-only, safe)

In [6]:
print("Pump station enabled    :", hipace450.get_pumpStatn_enabled())
print("Motor pump enabled      :", hipace450.get_motor_pump_enabled())
print("Standby mode            :", hipace450.get_standby())
print("Vent enabled            :", hipace450.get_vent_enabled())
print("Heating enabled         :", hipace450.get_heating_enabled())
print("Gas mode                :", hipace450.get_gas_mode(), "(0=heavy, 1=light, 2=He)")
print("Vent mode               :", hipace450.get_vent_mode(), "(0=delayed, 1=none, 2=direct; TC80 default 2)")

Pump station enabled    : False
Motor pump enabled      : True
Standby mode            : False
Vent enabled            : False
Heating enabled         : False
Gas mode                : 0 (0=heavy, 1=light, 2=He)
Vent mode               : 0 (0=delayed, 1=none, 2=direct; TC80 default 2)


In [13]:
print("Nominal speed           :", pump.get_nominal_speed_hz(), "Hz /", pump.get_nominal_speed_rpm(), "rpm")
print("Set speed               :", pump.get_set_speed_hz(), "Hz")
print("Actual speed            :", pump.get_actual_speed_hz(), "Hz /", pump.get_actual_speed_rpm(), "rpm")
print("Target speed reached    :", pump.is_target_speed_reached())
print("Pump accelerating       :", pump.is_pump_accelerating())
print("Rot. speed SwP reached  :", pump.get_rotationspd_SwP_reached())

Nominal speed           : 1500 Hz / 90000 rpm
Set speed               : 1500 Hz
Actual speed            : 0 Hz / 0 rpm
Target speed reached    : False
Pump accelerating       : False
Rot. speed SwP reached  : False


In [14]:
print("Drive current           :", pump.get_drive_current(), "A")
print("Drive voltage           :", pump.get_drive_voltage(), "V")
print("Drive power             :", pump.get_drive_power(), "W")
print("Temp electronics        :", pump.get_electronics_temperature(), "degC")
print("Temp pump bottom        :", pump.get_pump_bottom_temperature(), "degC")
print("Temp power stage (TC80) :", pump.get_power_stage_temperature(), "degC")
print("Temp rotor (TC80)       :", pump.get_rotor_temperature(), "degC")
print("Overtemp electronics    :", pump.is_overtemperature_electronics())
print("Overtemp pump           :", pump.is_overtemperature_pump())
print("Operating hours pump    :", pump.get_operating_hours_pump(), "h")
print("Operating hours elec    :", pump.get_operating_hours_electronics(), "h")
print("Pump cycles             :", pump.get_pump_cycles())
print("Accel/decel             :", pump.get_acceleration_deceleration(), "rpm/s")

Drive current           : 0.0 A
Drive voltage           : 24.08 V
Drive power             : 0 W
Temp electronics        : 31 degC
Temp pump bottom        : 26 degC
Temp power stage (TC80) : 30 degC
Temp rotor (TC80)       : 25 degC
Overtemp electronics    : False
Overtemp pump           : False
Operating hours pump    : 5 h
Operating hours elec    : 10 h
Pump cycles             : 0
Accel/decel             : 0 rpm/s


## 6. TC80 setpoints and configuration (read-only)

In [15]:
print("Ramp-up time setpoint   :", pump.get_ramp_up_time(), "min")
print("Speed setpoint          :", pump.get_speed_setpoint(), "%")
print("Power setpoint          :", pump.get_power_setpoint(), "%")
print("Speed set mode enabled  :", pump.get_speed_set_mode_enabled())
print("Temp management (TC80)  :", pump.get_temperature_management())
print("Max power output time   :", pump.get_max_power_output_time(), "s")
print("Fan-on temperature      :", pump.get_fan_on_temperature(), "degC")
print("Power output voltage    :", pump.get_power_output_voltage(), "V")
print("Power output threshold  :", pump.get_power_output_threshold(), "W")
print("Cfg accessory A1        :", pump.get_cfg_acc_a1())
print("Cfg accessory B1        :", pump.get_cfg_acc_b1())
print("Cfg accessory C1 (TC80) :", pump.get_cfg_acc_c1())
print("Cfg accessory D1 (TC80) :", pump.get_cfg_acc_d1())

Ramp-up time setpoint   : 8 min
Speed setpoint          : 65.0 %
Power setpoint          : 100 %
Speed set mode enabled  : False
Temp management (TC80)  : 0
Max power output time   : 10 s
Fan-on temperature      : 45 degC
Power output voltage    : 23.0 V
Power output threshold  : 20 W
Cfg accessory A1        : 2
Cfg accessory B1        : 13
Cfg accessory C1 (TC80) : 1
Cfg accessory D1 (TC80) : 13


In [14]:
pump.set_cfg_acc_a1(2)
pump.set_cfg_acc_b1(13)
pump.set_cfg_acc_d1(13)

## 7. Gauge readout (sensor on controller)

KB quirk: a deactivated gauge can return its last frozen value — always gate on `get_SensOnOff()`, not on value. 0.0 = never measured since power-up; ~1e80 = over-range sentinel (cold cathode at atmosphere).

In [22]:
sensor_on = pump.get_SensOnOff()
print(f"Gauge sensor on : {sensor_on}")

pressure = pump.get_gauge_pressure()
print(f"Gauge pressure  : {pressure:.2e} hPa")
if pressure == 0.0:
    print("  -> 0.0 = sensor off / never measured since power-up")
elif pump.data_converter.is_pressure_sentinel(pressure):
    print("  -> over-range sentinel (gauge at atmosphere?)")

Gauge sensor on : True
Gauge pressure  : 0.00e+00 hPa
  -> 0.0 = sensor off / never measured since power-up


In [21]:
# Switch the gauge sensor on/off if needed (uncomment):
pump.set_SensOnOff(True)
time.sleep(2)
print("Sensor on:", pump.get_SensOnOff())

Sensor on: True


In [25]:
pump.set_SensOnOff(False)

In [18]:
pump.get_SensOnOff()

True

In [19]:
# Continuous gauge readout: N reads, 1 s apart
N_READS = 20
for i in range(N_READS):
    p = pump.get_gauge_pressure()
    print(f"{time.strftime('%H:%M:%S')}  pressure = {p:.3e} hPa")
    time.sleep(1.0)

15:51:23  pressure = 1.275e+00 hPa
15:51:24  pressure = 1.276e+00 hPa
15:51:25  pressure = 1.277e+00 hPa
15:51:26  pressure = 1.275e+00 hPa
15:51:27  pressure = 1.279e+00 hPa
15:51:28  pressure = 1.274e+00 hPa
15:51:29  pressure = 1.279e+00 hPa
15:51:30  pressure = 1.278e+00 hPa
15:51:31  pressure = 1.279e+00 hPa
15:51:32  pressure = 1.277e+00 hPa
15:51:33  pressure = 1.279e+00 hPa
15:51:34  pressure = 1.280e+00 hPa
15:51:35  pressure = 1.277e+00 hPa
15:51:36  pressure = 1.279e+00 hPa
15:51:37  pressure = 1.279e+00 hPa
15:51:38  pressure = 1.281e+00 hPa
15:51:39  pressure = 1.279e+00 hPa
15:51:40  pressure = 1.280e+00 hPa
15:51:41  pressure = 1.280e+00 hPa
15:51:42  pressure = 1.279e+00 hPa


## 8. Convenience dictionaries

In [20]:
info = pump.get_system_info()
for k, v in info.items():
    print(f"{k:26s}: {v}")

omni_device_name          : OmC200
omni_serial_number        : 80116972
omni_firmware_version     : 010300
omni_hardware_version     : 010100
omni_rs485_address        : 100
omni_error_code           : 000000
pump_device_name          : TC 80
pump_firmware_version     : 010500
pump_rs485_address        : 1
pump_error_code           : 000000
pump_identification       : 30821
pressure                  : 1.281


In [21]:
status = pump.get_pump_status()
for k, v in status.items():
    print(f"{k:26s}: {v}")

actual_speed_hz           : 0
actual_speed_rpm          : 0
set_speed_hz              : 1500
drive_current             : 0.0
drive_voltage             : 24.08
drive_power               : 0
electronics_temp          : 32
pump_bottom_temp          : 26
power_stage_temp          : 30
rotor_temp                : 25
target_speed_reached      : False
pump_accelerating         : False
operating_hours           : 5


## 9. Housekeeping cycle test

One manual hk cycle, then a short threaded run. Output goes to the canonical log lines (console + `debugging/logs/`).

In [30]:
# Single housekeeping cycle (reads all HK_CHANNELS + gauge once)
pump.hk_monitor()

In [22]:
# Threaded housekeeping for 10 s
pump.start_housekeeping()
time.sleep(10)
pump.stop_housekeeping()
print("housekeeping stopped")

housekeeping stopped


## 10. Pump control (DANGER — only run deliberately)

All cells below CHANGE hardware state. Uncomment only what you intend.

Start/stop practice (KB, notebooks 007/010/022): start = `enable_motor_pump()` + `enable_pumpStatn()`; stop = `disable_pumpStatn()` (motor pump stays enabled). Manual-valve discipline: close the loadlock's HiPace80 valve before stop. Heating only accepted while pump runs (drive-electronics interlock).

In [ ]:
# START pump station
# pump.enable_motor_pump()
# pump.enable_pumpStatn()
# print("pump station enabled:", pump.get_pumpStatn_enabled())

In [ ]:
# Watch spin-up (run after starting)
# for i in range(30):
#     print(f"{time.strftime('%H:%M:%S')}  {pump.get_actual_speed_hz():4d} Hz  "
#           f"{pump.get_drive_current():.2f} A  {pump.get_drive_power():3d} W  "
#           f"accel={pump.is_pump_accelerating()}")
#     time.sleep(2)

In [ ]:
# STOP pump station (close the LL HiPace80 valve first!)
# pump.disable_pumpStatn()
# print("pump station enabled:", pump.get_pumpStatn_enabled())

In [ ]:
# Other state changes (uncomment as needed):
# pump.set_standby(True)          # / False
# pump.enable_vent()              # / pump.disable_vent()
# pump.enable_heating()           # only accepted while pump running
# pump.disable_heating()
# pump.acknowledge_error()

## 11. Disconnect

In [33]:
pump.connect()

True

In [34]:
pump.get_pump_status()

{'actual_speed_hz': 0,
 'actual_speed_rpm': 0,
 'set_speed_hz': 1500,
 'drive_current': 0.0,
 'drive_voltage': 24.07,
 'drive_power': 0,
 'electronics_temp': 37,
 'error': 'invalid checksum in gauge response'}

In [36]:
#See if RS485 is set

pump._query_channel_parameter(1, 60)

'002'

In [23]:
pump.stop_housekeeping()
pump.disconnect()
print("disconnected")

disconnected


In [44]:
hipace450.disconnect()

True